# Token gradients and activation patching

A tiny bidirectional transformer learns whether token 1 occurs in a length-six sequence. All tokens are synthetic IDs, with no tokenizer, padding, language pretraining, or causal mask. Derivatives are with respect to continuous embeddings, **not integer IDs**. Patching replaces one embedding activation from a matched clean example; it does not establish a unique circuit. Reference: [Vaswani et al. (2017)](https://arxiv.org/abs/1706.03762); intervention motivation: [Meng et al. (2022)](https://arxiv.org/abs/2202.05262).

Offline CPU tutorial. Fixed seeds and two PyTorch threads; small synthetic data only.


In [1]:
import os, sys, time, resource, json
from pathlib import Path
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root / "src"))
import numpy as np
import torch
from torch import nn
torch.set_num_threads(2)
torch.set_num_interop_threads(2)
torch.manual_seed(7)
np.random.seed(7)
started = time.perf_counter()
print("CPU only; torch", torch.__version__, "threads", torch.get_num_threads())



CPU only; torch 2.10.0.dev20251025+cpu threads 2


In [2]:
from autoexplain.toys import TinyTransformer, patch_token
model = TinyTransformer()
def sequences(count, seed):
    g = torch.Generator().manual_seed(seed)
    tokens = torch.randint(2, 12, (count, 6), generator=g)
    labels = torch.arange(count) % 2
    tokens[labels == 1, 2] = 1
    return tokens, labels
tokens, labels = sequences(48, 10)
optimizer = torch.optim.Adam(model.parameters(), lr=0.025)
for _ in range(45):
    optimizer.zero_grad()
    loss = nn.functional.cross_entropy(model(tokens), labels)
    loss.backward(); optimizer.step()
model.eval()
test, target = sequences(24, 91)
with torch.no_grad():
    accuracy = (model(test).argmax(1) == target).float().mean()
print("Held-out synthetic accuracy:", accuracy.item())
clean = test[1:2].clone()
corrupted = clean.clone(); corrupted[:, 2] = 2
embedded = model.embedding(clean).detach().requires_grad_(True)
score = model.forward_embeddings(embedded)[0, 1]
gradient = torch.autograd.grad(score, embedded)[0]
print("Per-token embedding gradient L2:", gradient.norm(dim=-1).tolist())
with torch.no_grad():
    donor = model.embedding(clean)
    clean_score = model(clean)[0, 1].item()
    baseline = model(corrupted)[0, 1].item()
    with patch_token(model.embedding, donor, token=2):
        patched = model(corrupted)[0, 1].item()
    with patch_token(model.embedding, model.embedding(corrupted), token=2):
        control = model(corrupted)[0, 1].item()
    restored = model(corrupted)[0, 1].item()
assert abs(restored - baseline) < 1e-6
assert abs(control - baseline) < 1e-6
assert abs(patched - clean_score) < 1e-5  # inputs differ at exactly this token
print({"clean": clean_score, "corrupt": baseline, "patched": patched, "self_patch_control": control})
print("Exact recovery here is expected from replacing the only changed embedding, not circuit discovery.")



Held-out synthetic accuracy: 1.0
Per-token embedding gradient L2: [[0.2989959418773651, 3.6832492351531982, 1.5374975204467773, 1.6077755689620972, 0.03933967277407646, 0.6207914352416992]]
{'clean': 4.512936592102051, 'corrupt': -5.767764091491699, 'patched': 4.512936592102051, 'self_patch_control': -5.767764091491699}
Exact recovery here is expected from replacing the only changed embedding, not circuit discovery.


In [3]:
elapsed = time.perf_counter() - started
rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
# Linux reports KiB; macOS reports bytes. This is this kernel's lifetime high-water mark.
peak_mib = rss / (1024 * 1024 if sys.platform == "darwin" else 1024)
print(json.dumps({"tutorial_compute_wall_seconds": round(elapsed, 3),
                  "kernel_lifetime_peak_rss_mib": round(peak_mib, 2),
                  "memory_scope": "kernel process only; includes imports; excludes child processes"}))



{"tutorial_compute_wall_seconds": 0.59, "kernel_lifetime_peak_rss_mib": 366.45, "memory_scope": "kernel process only; includes imports; excludes child processes"}
